# JCT-001 CPU engineering only
Enable Internet and the existing GITHUB_TOKEN secret. Run All once. No GPU or scientific training. Reports return to GitHub automatically; checkpoints stay in Output.


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

# CPU engineering only. Internet + existing GITHUB_TOKEN; Run All once.
# No GPU, model download, scientific Pilot or pytest suite.
SOURCE_SHA = 'd3e70fd4d6fb64f4faa103f71872d1770454323d'
REPOSITORY = "https://github.com/ArcheLabs/mini-cells.git"
CODE = Path("/kaggle/working") / ("jct001-engineering-code-" + SOURCE_SHA[:12])
OUTPUT = Path("/kaggle/working/jct001-engineering-output")
PATTERNS = ['/pyproject.toml', '/src/', '/shared/python/', '/research/projects/cellular-language-models/engineering/kaggle-continuation/', '/research/projects/cellular-language-models/experiments/006-path-first-clm/', '/research/projects/cellular-language-models/experiments/005-compositional-cell-graph/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r2/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/run.py', '/research/projects/continual-learning/context/04-continual-learning/scripts/research/', '/research/projects/cellular-language-models/context/08-hybrid-clm/scripts/research/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3b/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3c/moe-bridge/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3d/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3e/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-connection-admission/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3f-v2/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3g/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3h/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3i/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3j/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3k/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3l/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/implementation/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/artifact-recovery-001/independent-review-002/tokenizer.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-e/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f/audits/tiny-failure-001/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m-f-d1/', '/research/projects/cellular-language-models/experiments/jct-001/', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/GLOBAL_DATASET_USAGE_LEDGER.json', '/research/projects/cellular-language-models/experiments/clm-001-qfirst-integrated/r3m/registration/content-isolation-001/REVIEW.md']


def run(command, network=False):
    print("[JCT-001 CPU]", " ".join(map(str, command)), flush=True)
    return subprocess.run(list(map(str, command)), check=True, timeout=45 if network else None)


if not importlib.util.find_spec("tokenizers"):
    run([sys.executable, "-m", "pip", "install", "--quiet", "tokenizers==0.20.3"], network=True)
if not (CODE / ".git").exists():
    CODE.mkdir(parents=True, exist_ok=True)
    run(["git", "init", "--quiet", CODE])
    run(["git", "-C", CODE, "remote", "add", "origin", REPOSITORY])
    run(["git", "-C", CODE, "config", "remote.origin.promisor", "true"])
    run(["git", "-C", CODE, "config", "remote.origin.partialclonefilter", "blob:none"])
    run(["git", "-C", CODE, "sparse-checkout", "init", "--no-cone"])
    run(["git", "-C", CODE, "sparse-checkout", "set", "--no-cone", *PATTERNS])
    run(["git", "-C", CODE, "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_SHA], network=True)
    run(["git", "-C", CODE, "checkout", "--detach", "FETCH_HEAD"], network=True)
actual = subprocess.check_output(["git", "-C", str(CODE), "rev-parse", "HEAD"], text=True).strip()
if actual != SOURCE_SHA:
    raise RuntimeError("JCT_SOURCE_MISMATCH: use a fresh code folder")
print({"experiment": "JCT-001", "source_sha": actual, "scope": "SYNTHETIC_CPU_ENGINEERING_ONLY", "scientific_training_authorized": False}, flush=True)
try:
    run([sys.executable, CODE / "research/projects/cellular-language-models/experiments/jct-001/run.py",
         "--mode", "engineering", "--output", OUTPUT, "--publish-github"])
finally:
    print("Save Version retains CPU reports, publication receipt and local Checkpoints:", OUTPUT, flush=True)
